# Get Kegg Pathway Nodes
From the kegg pathways selected below filter out relevant edges and nodes, that are also represented in the L1000 database.
Write the available nodes to file in data_phase1 to use for further analysis

In [1]:
# --- Step 1: Download and parse multiple KEGG pathways ---

# make a pathway id dictionary with pathway_name as key and pathway_id as value
# pathway_id_dict = {
#     "MAPK_signaling": "hsa04010",
#     "T_cell_receptor_signaling": "hsa04660",
#     "TGF-beta_signaling": "hsa04350",
#     "p53_signaling": "hsa04115",
#     "Pathways_in_cancer": "hsa05200",
#     "mtor_signaling": "hsa04150",
#     "pi3k_akt_signaling": "hsa04151",
#     "apoptosis": "hsa04210",
#     "tnf_signaling": "hsa04668",
#     "nf_kb_signaling": "hsa04064",
#     "breast_cancer": "hsa05224",
#     "colorectal_cancer": "hsa05210",
#     "non_small_cell_lung_cancer": "hsa05223",
#     "small_cell_lung_cancer": "hsa05222",
#     "transcriptional_misregulation_in_cancer": "hsa05202",
#     "central_carbon_metabolism_in_cancer": "hsa05230",
#     "cell_cycle": "hsa04110",
#     "EGFR_signaling": "hsa01521"
# }

# read dictionary from kegg_data/hsa_pathways.json
import json
with open("kegg_data/hsa_pathways.json", "r") as f:
    pathway_id_dict = json.load(f)
pathway_id_dict

{'2_Oxocarboxylic_acid_metabolism': 'hsa01210',
 'ABC_transporters': 'hsa02010',
 'AGE_RAGE_signaling_in_diabetic_complications': 'hsa04933',
 'AMPK_signaling': 'hsa04152',
 'ATP_dependent_chromatin_remodeling': 'hsa03082',
 'Acute_myeloid_leukemia': 'hsa05221',
 'Adherens_junction': 'hsa04520',
 'Adipocytokine_signaling': 'hsa04920',
 'Adrenergic_signaling_in_cardiomyocytes': 'hsa04261',
 'African_trypanosomiasis': 'hsa05143',
 'Alanine_aspartate_and_glutamate_metabolism': 'hsa00250',
 'Alcoholic_liver': 'hsa04936',
 'Alcoholism': 'hsa05034',
 'Aldosterone_regulated_sodium_reabsorption': 'hsa04960',
 'Aldosterone_synthesis_and_secretion': 'hsa04925',
 'Allograft_rejection': 'hsa05330',
 'Alzheimer': 'hsa05010',
 'Amino_sugar_and_nucleotide_sugar_metabolism': 'hsa00520',
 'Aminoacyl_tRNA_biosynthesis': 'hsa00970',
 'Amoebiasis': 'hsa05146',
 'Amphetamine_addiction': 'hsa05031',
 'Amyotrophic_lateral_sclerosis': 'hsa05014',
 'Antifolate_resistance': 'hsa01523',
 'Antigen_processing_and_

In [2]:
from Bio.KEGG.KGML import KGML_parser
from Bio.KEGG import REST
from io import StringIO
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import os

In [3]:
def get_geneid_mapping():
    """
    Fetch gene symbols for a list of KEGG gene IDs.
    """
    df_gene_info = pd.read_csv('data_phase1/gene_info.txt', sep='\t')  
    # create a map by using the columns 'gene_id' and 'gene_symbol'
    L1000_gene_map = df_gene_info.set_index('pr_gene_id')['pr_gene_symbol'].to_dict()
    # to every gene_id in the mapping add "hsa:" as a prefix
    L1000_gene_map = {f"hsa:{k}": v for k, v in L1000_gene_map.items()}
    return L1000_gene_map

In [4]:
def download_pathway(pathway_id):
    kgml = REST.kegg_get(pathway_id, "kgml").read()
    return KGML_parser.read(StringIO(kgml))

In [5]:
pathways = pathway_id_dict.keys()
NETWORK_NAME = "TP53"
base_node= "TP53" #node that should define the subgraph, if none largest component is used

pathways = [download_pathway(pathway_id) for pathway_id in [
    pathway_id_dict[pathway] for pathway in pathways
]]



In [6]:
import os
import json
import networkx as nx
from pathlib import Path

# ------------------------------------------------------------
# Assumed to be defined already in your environment:
# - pathways: iterable of KEGG pathway objects
# - gene_id_to_symbol: dict mapping 'hsa:XXXX' -> 'GENE'
# - activation, inhibition, other: sets of KEGG relation subtype strings
# - kegg_nodes_path: path to JSON file
# ------------------------------------------------------------


gene_id_to_symbol = get_geneid_mapping()
kegg_nodes_path = Path("kegg_data/kegg_nodes.json")
id_to_human = {v: k for k, v in pathway_id_dict.items()}
parent_dir = kegg_nodes_path.parent
# Fail fast if target directory does not exist
if not parent_dir.exists():
    raise FileNotFoundError(f"Required directory does not exist: {parent_dir}. Create it before running this cell.")


activation = ["activation", "expression"] # self loops should not be expression
inhibition = ["inhibition", "repression"]
other = ["compound", "hidden compound", "indirect effect", "state change", "binding/association", 
         "dissociation", "missing interaction", "phosphorylation", "dephosphorylation", "glycosylation", 
         "ubiquitination", "methylation"]

# --- Load existing JSON if it exists (do NOT overwrite) ---
if os.path.exists(kegg_nodes_path):
    with open(kegg_nodes_path, "r") as f:
        data = json.load(f)
else:
    data = {}

# --- Initialize global graph ---
G_all = nx.DiGraph()

added_genes = 0
activating_edges = 0
inhibiting_edges = 0
skipped_edges = 0

# ============================================================
# Build combined graph from all pathways
# ============================================================
for pw in pathways:
    # KEGG entry ID -> list of valid gene IDs
    id_to_gene_ids = {}

    # --- Add gene nodes ---
    for gene in pw.genes:
        gene_ids = gene.name.split()
        valid_gene_ids = [gid for gid in gene_ids if gid in gene_id_to_symbol]

        if not valid_gene_ids:
            continue

        id_to_gene_ids[gene.id] = valid_gene_ids

        for gid in valid_gene_ids:
            symbol = gene_id_to_symbol[gid]
            if symbol not in G_all:
                G_all.add_node(symbol, type="gene")
                added_genes += 1

    # --- Add edges ---
    for rel in pw.relations:
        src_id = rel.entry1.id
        tgt_id = rel.entry2.id

        if src_id not in id_to_gene_ids or tgt_id not in id_to_gene_ids:
            continue

        src_genes = id_to_gene_ids[src_id]
        tgt_genes = id_to_gene_ids[tgt_id]

        for subtype in rel.subtypes:
            interaction = subtype[0]

            if interaction in activation:
                interaction_code = "1"
                activating_edges += len(src_genes) * len(tgt_genes)
            elif interaction in inhibition:
                interaction_code = "2"
                inhibiting_edges += len(src_genes) * len(tgt_genes)
            elif interaction in other:
                skipped_edges += len(src_genes) * len(tgt_genes)
                continue
            else:
                skipped_edges += len(src_genes) * len(tgt_genes)
                continue

            # same direction as in your original code
            for src_gid in src_genes:
                for tgt_gid in tgt_genes:
                    src_symbol = gene_id_to_symbol[src_gid]
                    tgt_symbol = gene_id_to_symbol[tgt_gid]

                    G_all.add_edge(
                        tgt_symbol,
                        src_symbol,
                        interaction=interaction_code
                    )

# ============================================================
# Global postprocessing
# ============================================================
print(f"✅ Added {added_genes} genes before cleanup.")
print(f"✅ Raw combined graph: {G_all.number_of_nodes()} nodes, {G_all.number_of_edges()} edges.")

# --- Remove isolated nodes ---
G_all.remove_nodes_from(list(nx.isolates(G_all)))
print(f"✅ After removing isolates: {G_all.number_of_nodes()} nodes, {G_all.number_of_edges()} edges.")

if G_all.number_of_nodes() == 0 or G_all.number_of_edges() == 0:
    raise ValueError("Combined graph is empty after removing isolates.")

# --- Largest weakly connected component ---
wcc = list(nx.weakly_connected_components(G_all))
largest_wcc = max(wcc, key=len)
G_all = G_all.subgraph(largest_wcc).copy()

print(f"✅ Largest WCC selected: {G_all.number_of_nodes()} nodes, {G_all.number_of_edges()} edges.")
print(
    f"✅ {activating_edges} activating, "
    f"{inhibiting_edges} inhibiting, "
    f"{skipped_edges} skipped edges."
)

# ============================================================
# Store combined entry in JSON (append, do not overwrite others)
# ============================================================
data["combined"] = {
    "nodes": list(G_all.nodes()),
    "edges": G_all.number_of_edges()
}

with open(kegg_nodes_path, "w") as f:
    json.dump(data, f, indent=2, sort_keys=True)

print(f"✅ Appended 'combined' entry to {kegg_nodes_path}")

# ============================================================
# Save combined graph object
# ============================================================
import pickle

combined_graph_path = "kegg_combined_graph.pkl"

with open(combined_graph_path, "wb") as f:
    pickle.dump(G_all, f)

print(f"✅ Saved combined graph to {combined_graph_path}")


✅ Added 6333 genes before cleanup.
✅ Raw combined graph: 6333 nodes, 20635 edges.
✅ After removing isolates: 2846 nodes, 20635 edges.
✅ Largest WCC selected: 2710 nodes, 20518 edges.
✅ 44023 activating, 9629 inhibiting, 38243 skipped edges.
✅ Appended 'combined' entry to kegg_data/kegg_nodes.json
✅ Saved combined graph to kegg_combined_graph.pkl


In [7]:
def create_topo_file_from_graph(network_name, G: nx.DiGraph, dir):
    """
    Create a topo file as expected by racipe from a nx Graph
    and store it in the const.TOPO_PATH directory.
    :param G: nx Graph
    """
    new_file_path = Path(dir) / f"{network_name}.topo" 
    # save graph to a trrust.topo file with the header Source Target Type
    with open(new_file_path, "w") as f:
        f.write("Source Target Type\n")
        for u, v, d in G.edges(data='interaction'):
            f.write(f"{u} {v} {d}\n")
    print(f"✅ Saved topo file for graph with {G.number_of_edges()} edges  and {G.number_of_nodes()} nodes to {new_file_path}")

create_topo_file_from_graph("combined", G_all, "experiment_data")

✅ Saved topo file for graph with 20518 edges  and 2710 nodes to experiment_data/combined.topo


In [9]:
# L1000 to id
L1000_to_idx_map = {gene: idx for idx, gene in enumerate(G_all.nodes())}

In [10]:
def kegg_to_L1000():
    """
    Fetch gene symbols for a list of KEGG gene IDs.
    """
    df_gene_info = pd.read_csv('data_phase1/gene_info.txt', sep='\t')  
    # create a map by using the columns 'gene_id' and 'gene_symbol'
    L1000_gene_map = df_gene_info.set_index('pr_gene_id')['pr_gene_symbol'].to_dict()
    # to every gene_id in the mapping add "hsa:" as a prefix
    L1000_gene_map = {f"hsa:{k}": v for k, v in L1000_gene_map.items()}
    return L1000_gene_map

kegg_to_L1000_map = kegg_to_L1000()

In [11]:
# store L1000_to_idx_map as gene_to_idx.pkl
import pickle
with open(f"experiment_data/gene_to_idx.pkl", "wb") as f:
    pickle.dump(L1000_to_idx_map, f)
print(f"✅ Saved gene to index mapping to experiment_data/gene_to_idx.pkl")
with open(f"experiment_data/kegg_to_L1000_map.pkl", "wb") as f:
    # from the keys of kegg_to_L1000_map cut the "hsa:" prefix
    kegg_to_L1000_map_no_prefix = {k.split("hsa:")[-1]: v for k, v in kegg_to_L1000_map.items()}
    pickle.dump(kegg_to_L1000_map_no_prefix, f)
print(f"✅ Saved gene to index mapping to experiment_data/kegg_to_L1000_map.pkl")


✅ Saved gene to index mapping to experiment_data/gene_to_idx.pkl
✅ Saved gene to index mapping to experiment_data/kegg_to_L1000_map.pkl
